In [ ]:
import time
import json
from datetime import datetime
from conexion_host import ConexionHost

INTERVALO_POLLING = 0.2

def main():
    config={
        "host":"tic-host.local",
         "port":22,
         "username":"grupo02",
         "password":"qjMv6LXVyxP6A1MZ",
         "remote_inbox":"/inbox",
         "remote_outbox":"/outbox"
         }
    
    player_id = "P02"
    
    cliente = ConexionHost(config)
    
    try:
        print(f"conectando con el host central ({config['host']})...")
        cliente.conectar()
    except Exception as e:
        print(f"error critico al conectar:{e}")
        return
    
    event_id = f"{player_id}-0001"
    mensaje_join = {
        "timestamp": datetime.now().isoformat(),
        "player_id": player_id,
        "Action": "Join",
        "EventID": event_id,
        "Stage": "Lobby"
        }
    try:
        cliente.enviar_json(mensaje_join)
        print(f"Mensaje Join enviado con EventID: {event_id}")
    except Exception as e:
        print(f"No se pudo enviar el mensaje Join: {e}")
        cliente.cerrar_conexion()
        return
    print("Esperando confirmacion (Accepted) en el Lobby[cite: 6]...")
    registrado = False
    
    while not registrado:
        try:
            mensajes = cliente.leer_nuevos_json()
            for msg in mensajes:
                if msg.get("Action") == "Accepted" and msg.get("ReplyTo") == event_id:
                    print(f"¡Ingreso al Lobby exitoso! Respuesta del host: {msg}")
                    registrado = True
                    break
        except Exception as e:
            print(f"Error al leer la outbox: {e}")
        
        if not registrado:
            time.sleep(INTERVALO_POLLING)
            
    print("jugador lsto y a la escucha de instrucciones del servidor...")
    
    try:
        while True:
            mensaje = cliente.leer_nuevos_json()
            for msg in mensajes:
                print(f"mensaje recibido: {msg}")
                
            time.sleep(INTERVALO_POLLING)
    except KeyboardInterrupt:
        print("\nCerrando cliente de forma ordenada...")
        cliente.cerrar_conexion()
        
        
if __name__ == "__main__":
        main()
